## 1. HTML for Data Science

HTML (HyperText Markup Language) is the standard language used to create web pages and structure content on the web.

Basic syntax:

```html
<tagname>content</tagname>
```

Basic structure:

```html
<!DOCTYPE html>
<html>
<head>
    <title>My First Page</title>
</head>
<body>
    <h1>Welcome to PRIME!</h1>
    <p>This is a paragraph of text.</p>
</body>
</html>
```

Important elements from the course:
- `<h1>` to `<h6>` — headings
- `<p>` — paragraphs
- `<a>` — links; useful for extracting URLs from `href`
- `<img>` — images; useful for extracting URLs from `src`
- `<table>` — tabular data
- `<tr>` — table row
- `<td>` / `<th>` — table cells
- `<ul>` / `<ol>` / `<li>` — lists
- `<div>` / `<span>` — generic containers
- `<form>` / `<button>` / `<input>` — form-related elements


In [ ]:
html = """
<!DOCTYPE html>
<html>
<head>
    <title>My First Page</title>
</head>
<body>
    <h1>Welcome to PRIME!</h1>
    <p>This is a paragraph of text.</p>
</body>
</html>
"""

print(html)


## 2. HTML Attributes

Attributes provide additional information about HTML elements.

Common attributes used during scraping:

| Attribute | Typical use |
|---|---|
| `href` | URL in an `<a>` tag |
| `src` | Source URL in an `<img>` tag |
| `id` | Unique identifier |
| `class` | Grouping/styling; heavily used in scraping |
| `type` | Type of form input |
| `name` | Name assigned to a form element |
| `value` | Data associated with a form element |

Example:

```html
<a href="https://example.com" id="link1" class="external">
    Visit Example
</a>
```


## 3. HTML Tables and Forms

### Table

```html
<table>
    <tr>
        <th>Name</th>
        <th>Marks</th>
    </tr>
    <tr>
        <td>Adam</td>
        <td>95</td>
    </tr>
    <tr>
        <td>Bob</td>
        <td>88</td>
    </tr>
</table>
```

### Form

```html
<form action="/submit" method="POST">
    <label for="name">Name:</label>
    <input type="text" id="name" name="name">
    <input type="submit" value="Submit">
</form>
```


## 4. Web Scraping

The course introduces three commonly used Python tools:

1. **`requests`** — downloads webpages / sends HTTP requests.
2. **`BeautifulSoup`** — parses and extracts information from HTML; especially useful for static HTML.
3. **`Selenium`** — automates a browser and can be used for dynamically rendered JavaScript pages.



In [ ]:
# Install if needed:
# %pip install requests beautifulsoup4 lxml pandas

import requests
from bs4 import BeautifulSoup
import pandas as pd


## 5. `requests`

The `requests` module lets Python send HTTP requests to websites and APIs.

Basic GET request:

```python
import requests

URL = "https://example.com"
res = requests.get(URL)

print(res.status_code)
print(res.text)
print(res.content)
print(res.headers)
```

Important response attributes:
- `status_code` — HTTP status code
- `text` — response body as a string
- `content` — raw response bytes
- `headers` — response metadata
- `json()` — converts a JSON response into Python objects


In [ ]:
import requests

URL = "https://example.com"

res = requests.get(URL)

print("Status:", res.status_code)
print("Content-Type:", res.headers.get("Content-Type"))
print("First 200 characters:")
print(res.text[:200])


## 6. Common HTTP Status Codes

| Code | Meaning |
|---:|---|
| `200` | OK |
| `301` / `302` | Redirect |
| `400` | Bad Request |
| `401` | Unauthorized |
| `403` | Forbidden |
| `404` | Not Found |
| `500` | Server Error |

Always check the response status before processing data.


In [ ]:
URL = "https://example.com"
res = requests.get(URL)

if res.status_code == 200:
    print(res.text[:500])
else:
    print("Failed to fetch page:", res.status_code)


## 7. Timeouts and Exception Handling

A request can fail because of connection errors, timeouts, DNS issues, or HTTP errors.

The course demonstrates:

```python
try:
    r = requests.get("https://example.com", timeout=5)
    r.raise_for_status()
except requests.exceptions.RequestException as e:
    print("Error:", e)
```

`raise_for_status()` raises an exception for unsuccessful HTTP responses.


In [ ]:
try:
    r = requests.get("https://example.com", timeout=5)
    r.raise_for_status()
    print("Request successful:", r.status_code)
except requests.exceptions.RequestException as e:
    print("Error:", e)


## 8. Headers

Some websites expect headers such as `User-Agent`.

The course example:

```python
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(
    "https://example.com",
    headers=headers
)
```

Headers can help a server understand what kind of client is making the request.

Also use reasonable delays when scraping so that you do not send requests too rapidly.


In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(
    "https://example.com",
    headers=headers,
    timeout=10
)

print(response.status_code)


## 9. Download and Store HTML

The course stores downloaded HTML in a `scraped_data` directory:

```python
response = requests.get("https://example.com")

with open("scraped_data/data.html", "w") as f:
    f.write(response.text)
```

### Important path concept

A relative path such as:

```python
"scraped_data/data.html"
```

is relative to Python's **current working directory**, not automatically to the `.py` file.

For a robust project path, use `pathlib` and anchor it to the Python file:

```python
from pathlib import Path

BASE_DIR = Path(__file__).resolve().parent
output_path = BASE_DIR / "scraped_data" / "data.html"
```

For a Jupyter notebook, `Path.cwd()` shows the notebook's current working directory.


In [ ]:
from pathlib import Path

print("Current working directory:", Path.cwd())


## 10. API Data Collection

The same `requests` library can retrieve API responses.

Example API workflow:

```python
URL = "https://stephen-king-api.onrender.com/api/books"
res = requests.get(URL)

json_data = res.json()
```

JSON data can then be normalized into a Pandas DataFrame.


In [ ]:
URL = "https://stephen-king-api.onrender.com/api/books"

res = requests.get(URL, timeout=10)
print(res.status_code)

if res.status_code == 200:
    json_data = res.json()
    print(type(json_data))
    print(json_data)
else:
    print("Request failed:", res.status_code)


In [ ]:
# Normalize API JSON into a DataFrame
if res.status_code == 200:
    df_api = pd.json_normalize(json_data["data"])
    print(df_api.columns.tolist())

    # Select the fields used in the course practical, if available.
    expected = ["id", "Title", "Year", "ISBN"]
    available = [col for col in expected if col in df_api.columns]

    df_api = df_api[available]
    display(df_api)


## 11. BeautifulSoup

BeautifulSoup (`bs4`) parses HTML and XML documents.

Install:

```bash
pip install beautifulsoup4
```

Import:

```python
from bs4 import BeautifulSoup
```

The course uses:

```python
soup = BeautifulSoup(html_content, "lxml")
```

BeautifulSoup creates a **parse tree** from the HTML. You can navigate the tree to extract:
- Text
- Attributes
- Links
- Images
- Tables
- Other HTML elements


In [ ]:
from bs4 import BeautifulSoup

sample_html = """
<html>
    <head>
        <title>Example</title>
    </head>
    <body>
        <h1>Hello</h1>
        <p>This is a paragraph.</p>
    </body>
</html>
"""

soup = BeautifulSoup(sample_html, "lxml")

print(soup.title.get_text(strip=True))
print(soup.h1.get_text(strip=True))
print(soup.p.get_text(strip=True))


## 12. Useful BeautifulSoup Selection Methods

```python
soup.find("h1")
soup.find_all("h3")

element.get_text(strip=True)

element["href"]
element["class"]

element.find_parent("div")
element.find_next("div")

element.select("span.country-population")
element.select_one("span.country-population")
```

### Important distinction

- `find()` → first matching element
- `find_all()` → all matching elements
- `select()` → all elements matching a CSS selector
- `select_one()` → first element matching a CSS selector
- `get_text(strip=True)` → clean text from an element


## 13. Module 18 Practical — Scrape Countries

The provided course notebook uses:

```text
https://www.scrapethissite.com/pages/simple/
```

The workflow is:

1. Send a GET request.
2. Check the response.
3. Save the HTML.
4. Read the saved HTML.
5. Parse it with BeautifulSoup.
6. Find country headings.
7. Extract country names and populations.
8. Store the results in a list.
9. Convert the list into a DataFrame.
10. Save the DataFrame as CSV.


In [ ]:
import requests

URL = "https://www.scrapethissite.com/pages/simple/"

res = requests.get(
    URL,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=10
)

print("Status:", res.status_code)
print("Content-Type:", res.headers.get("Content-Type"))


In [ ]:
# Save the downloaded HTML.
# This version creates the directory automatically.

from pathlib import Path

data_dir = Path.cwd() / "scraped_data"
data_dir.mkdir(parents=True, exist_ok=True)

html_path = data_dir / "data1.html"

if res.status_code == 200:
    html_path.write_text(res.text, encoding="utf-8")
    print("Saved:", html_path.resolve())
else:
    print("Failed to fetch page:", res.status_code)


In [ ]:
# Read and parse the saved HTML.

html_content = html_path.read_text(encoding="utf-8")

soup = BeautifulSoup(html_content, "lxml")

print(soup.title.get_text(strip=True) if soup.title else "No title")


In [ ]:
# Extract countries and populations.

all_h3 = soup.find_all("h3")

all_countries = []

for h3 in all_h3:
    name = h3.get_text(strip=True)

    population_node = h3.find_next("div").select_one(
        "span.country-population"
    )

    if population_node:
        population = population_node.get_text(strip=True)
        all_countries.append([name, population])

print("Countries collected:", len(all_countries))
print(all_countries[:5])


In [ ]:
# Convert scraped data to Pandas.

df = pd.DataFrame(
    all_countries,
    columns=["Name", "Population"]
)

display(df.head())


In [ ]:
# Save cleaned data as CSV.

cleaned_dir = Path.cwd() / "cleaned_data"
cleaned_dir.mkdir(parents=True, exist_ok=True)

csv_path = cleaned_dir / "data.csv"

df.to_csv(csv_path, index=False)

print("Saved:", csv_path.resolve())


## 14. Inspecting the Scraped Result

Always inspect the output before considering a scraping task complete.

Useful checks:

```python
df.head()
df.tail()
df.shape
df.columns
df.info()
df.isna().sum()
```


In [ ]:
display(df.head())
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.info()


## 15. Extracting Links and Attributes

Links:

```python
links = soup.find_all("a")

for link in links:
    text = link.get_text(strip=True)
    href = link.get("href")
```

Images:

```python
images = soup.find_all("img")

for image in images:
    src = image.get("src")
```

Using `.get()` is useful because the attribute may not exist.


In [ ]:
# Small example using the sample HTML.
sample = BeautifulSoup("""
<div>
    <a href="https://example.com">Example</a>
    <img src="/images/logo.png">
</div>
""", "lxml")

for link in sample.find_all("a"):
    print(link.get_text(strip=True), "->", link.get("href"))

for image in sample.find_all("img"):
    print("Image:", image.get("src"))


## 16. Scraping Tables

HTML tables commonly use:

```html
<table>
    <tr>
        <th>Name</th>
        <th>Marks</th>
    </tr>
    <tr>
        <td>Adam</td>
        <td>95</td>
    </tr>
</table>
```

For simple tables, Pandas can sometimes read them directly:

```python
tables = pd.read_html(url)
```

For more control, parse the table with BeautifulSoup and explicitly extract rows/cells.


## 17. Static vs Dynamic Websites

### Static HTML
If the required data is already present in the HTML response, a common workflow is:

```text
requests → HTML → BeautifulSoup → extract data
```

### Dynamic JavaScript content
If the data is loaded after page rendering through JavaScript, the initial `requests.get()` response may not contain the desired data.

The course mentions **Selenium** as an optional browser automation alternative for such pages.

```text
Browser automation → rendered page → extraction
```

Do not assume every website requires Selenium; first inspect the actual HTTP response and page source.


## 18. API vs Web Scraping

### API
You request structured data from an endpoint.

```text
Python → API → JSON → Pandas
```

Advantages:
- Usually structured
- Easier to parse
- Often more stable than HTML selectors

### Web Scraping
You download a webpage and extract information from its HTML.

```text
Python → HTML → BeautifulSoup → extracted data → Pandas
```

The correct approach depends on what the website/API provides.
